In [ ]:
import pandas as pd
import requests
import re
from tqdm import tqdm  # optional, for progress bar

# ---------- 1. API configuration ----------
API_KEY = "sk-65a6dd459e744a4198d4435651a10399"  # Consider using environment variable
API_URL = "https://api.deepseek.com/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# System prompt (the rubric)
SYSTEM_PROMPT = """You are a research assistant specialized in financial language analysis. Your task is to classify excerpts from earnings calls as Clear or Obfuscatory using the structured rubric below. Only label the text that appears in the excerpt (which should be management’s spoken words). Apply the rubric consistently.

Rubric (5 dimensions, each scored on a continuous scale from 0 to 1):

1. Numerical Specificity
   Score 0: Contains specific numbers, percentages, dates, or explicit ranges (e.g., "revenue increased 12%", "by March 2025", "EPS $1.22–1.28").
   Score 0.5: Has some numbers but remains vague (e.g., "high single‑digit growth", "over 10%", "double‑digit increase").
   Score 1: No numbers; relies on vague quantifiers (e.g., "strong growth", "substantial improvement", "meaningful traction").
   → Assign any value between 0 and 1 based on how close the excerpt is to these anchors.

2. Quantification / Measurability
   Score 0: Exact absolute or per‑unit values (e.g., "$1.2B", "15 million users", "350 basis points").
   Score 0.5: Comparative or relative terms without full context (e.g., "up from last year", "margin improved", "better than expected").
   Score 1: No quantification; purely qualitative statements (e.g., "strong performance", "well‑positioned", "attractive opportunities").
   → Assign any value between 0 and 1.

3. Jargon & Buzzwords
   Score 0: Plain language, everyday business terms (e.g., "revenue", "customers", "employees", "cost").
   Score 0.5: Occasional buzzwords but still conveys specific meaning (e.g., "achieved synergies of $50M").
   Score 1: Heavy jargon or empty buzzwords without concrete meaning (e.g., "strategic initiatives", "value proposition", "ecosystem", "leverage", "tailwinds", "holistic approach").
   → Assign any value between 0 and 1.

4. Hedging / Certainty
   Score 0: Direct, no qualifiers (e.g., "will", "expect", "are", "did", "our guidance is").
   Score 0.5: Moderate hedging (e.g., "expect to", "anticipate", "believe", "likely").
   Score 1: Heavy hedging / evasion (e.g., "we believe", "we think", "it appears", "potentially", "could", "if conditions permit").
   → Assign any value between 0 and 1.

5. Actionability / Concreteness
   Score 0: Specific actions taken, clear causal links (e.g., "due to", "by doing X", "we launched", "we invested $X to achieve Y").
   Score 0.5: Mentions actions without specifics (e.g., "we are investing", "initiatives are underway", "we are focusing on").
   Score 1: Abstract, no actions (e.g., "momentum", "strength", "positioning", "confidence").
   → Assign any value between 0 and 1.


Decision Rule:
- Sum the five continuous dimension scores → Total Score (range 0–5).
- Total Score <= 2.5 → Label = 0
- Total Score > 2.5 → Label = 1

Output Format:
Dimension Scores:
1. Numerical Specificity: [score]
2. Quantification: [score]
3. Jargon: [score]
4. Hedging: [score]
5. Actionability: [score]
Total Score: [sum]
Label = [0,1]

Now, classify the following excerpt:

Excerpt:

"""

In [ ]:
def classify_excerpt(text):
    """Send one excerpt to DeepSeek API and return the full response text."""
    if not text or len(text.strip()) == 0:
        return "Empty excerpt"

    user_prompt = f"Now, classify the following excerpt:\n\nExcerpt:\n{text}"

    payload = {
        "model": "deepseek-chat",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.0,
        "max_tokens": 1000
    }

    try:
        response = requests.post(API_URL, headers=headers, json=payload, timeout=60)
        response.raise_for_status()
        result = response.json()
        content = result['choices'][0]['message']['content']
        return content
    except Exception as e:
        return f"Error: {str(e)}"

def extract_total_score(response_text):
    """
    Extract total score from the API response text.
    Returns a float if found, otherwise None.
    """
    # Match either **Total Score:** or Total Score: (with optional spaces)
    pattern = r"\*?\*?Total Score:\*?\*?\s*(\d+\.?\d*)"
    match = re.search(pattern, response_text, re.IGNORECASE)
    if match:
        try:
            return float(match.group(1))
        except ValueError:
            return None
    return None

# ---------- 2. Read the Excel file ----------
file_path = "full excerpts input.xlsx"   # <-- CHANGE to your actual file path
df = pd.read_excel(file_path)

# ---------- 3. Identify the column containing answer texts ----------
answer_column = "text"   # <-- CHANGE to your column name if different
if answer_column not in df.columns:
    raise ValueError(f"Column '{answer_column}' not found. Available columns: {list(df.columns)}")

# ---------- 4. Add new columns for results ----------
df['full_output'] = ""
df['total_score'] = None
df['binary_label'] = -1  # -1 indicates error or missing

# ---------- 5. Loop through each row and classify ----------
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying"):
    text = row[answer_column]
    if text and isinstance(text, str) and len(text.strip()) > 0:
        output = classify_excerpt(text)
        df.at[idx, 'full_output'] = output

        # Extract total score
        score = extract_total_score(output)
        if score is not None:
            df.at[idx, 'total_score'] = score
            df.at[idx, 'binary_label'] = 0 if score <= 2.5 else 1
        else:
            # Score extraction failed
            df.at[idx, 'binary_label'] = -1

        # Optional: print first 3 outputs for debugging
        if idx < 3:
            print(f"\n=== Row {idx} output ===\n{output}\n")
            print(f"Extracted score: {score}\n")
    else:
        df.at[idx, 'full_output'] = "No text found in column"
        df.at[idx, 'binary_label'] = -1

# ---------- 6. Save results ----------
output_path = "res.xlsx"
df.to_excel(output_path, index=False)
print(f"✅ Results saved to {output_path}")
print("\nFirst few rows of new columns:")
print(df[['text', 'total_score', 'binary_label']].head())

Classifying:   0%|          | 1/628 [00:17<3:03:48, 17.59s/it]


=== Row 0 output ===
Dimension Scores:  
1. Numerical Specificity: 0.8  
2. Quantification: 0.7  
3. Jargon: 0.6  
4. Hedging: 0.4  
5. Actionability: 0.3  
Total Score: 2.8  
Label = 1  

**Explanation:**  
- **Numerical Specificity (0.8):** The excerpt lacks specific numbers or percentages, using vague terms like "pretty high," "increasing steadily," "manyfold," and "more than twice." It mentions metrics (percentage time/mileage) but provides no exact figures.  
- **Quantification (0.7):** While it references metrics (e.g., "percentage time used," "daily active rate"), it does not give exact values, relying on relative comparisons ("increased manyfold," "more than twice") without baseline context.  
- **Jargon (0.6):** Contains some industry-specific terms like "autonomous driving," "HD mapless NOA," and "adoption rate," but these are not overly abstract and are necessary for the topic. However, phrases like "investment yield" and "operating metrics" lean toward buzzword usage.  
- 

Classifying:   0%|          | 2/628 [00:28<2:21:08, 13.53s/it]


=== Row 1 output ===
Dimension Scores:  
1. Numerical Specificity: 0.2 (Specific percentage "70%" and price threshold "RMB300,000" are given, but other parts lack numbers)  
2. Quantification: 0.3 (Includes a specific percentage and price point; other statements are comparative/qualitative but still grounded in measurable concepts)  
3. Jargon: 0.7 (Contains buzzwords like "entry barriers," "AI-powered autonomous driving," "computing power," "training capability," though some are industry-specific terms with implied meaning)  
4. Hedging: 0.6 (Uses "we believe" twice, which introduces moderate hedging, but other statements are presented as factual connections)  
5. Actionability: 0.4 (Describes a causal loop—improvements driving sales and investment—but lacks specific actions or concrete steps taken)  
Total Score: 2.2  
Label = 0

Extracted score: 2.2



Classifying:   0%|          | 3/628 [00:37<2:01:15, 11.64s/it]


=== Row 2 output ===
Dimension Scores:  
1. Numerical Specificity: 0.2 (Specific number "18.7%" is given, but also vague "around 18%")  
2. Quantification: 0.2 (Exact percentage "18.7%" and relative comparison "around 18%")  
3. Jargon: 0.6 (Some buzzwords like "snowball effect," "coexist," "R&D system," "methodologies," "product mix," but still conveys meaning)  
4. Hedging: 0.4 (Moderate hedging with "our view is that we will," "I think," "around 18%")  
5. Actionability: 0.5 (Mentions actions like "learn from Huawei" and "effort of the company," but not highly specific)  
Total Score: 1.9  
Label = 0

Extracted score: 1.9



Classifying: 100%|██████████| 628/628 [1:37:31<00:00,  9.32s/it]

✅ Results saved to res.xlsx

First few rows of new columns:
                                                text total_score  binary_label
0  Since the beginning, our investment yield on a...         2.8             1
1  And the percentage of NOA AD Max take rate on ...         2.2             0
2  So this is a very positive snowball effect. (F...         1.9             0
3  So for the third quarter, we believe our vehic...         2.2             0
4  And the second one is to make sure that we hav...         2.5             0
